## GPU availability

In [ ]:
!nvidia-smi

## Connect to Google Drive

Connect to your Google Drive. We'll work directly in the Drive folder to keep everything persistent.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Setup Project Directory in Drive

Set the path to your project folder in Google Drive. We'll work directly here - no copying needed!

In [ ]:
# Set your Google Drive folder path where the project will be located
# Example: '/content/drive/MyDrive/BEFUnet'
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/BEFUnet'  # Change this to your path

# Create directory if it doesn't exist
import os
os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)

# Change to Drive project directory - we'll work here directly
os.chdir(DRIVE_PROJECT_PATH)
print(f"Working directory: {os.getcwd()}")

## Clone the BEFUnet repository

Clone directly into the Drive folder.

In [ ]:
# Clone repository directly into Drive folder
# If repo already exists, this will fail - that's OK, just continue
!git clone https://github.com/Omid-Nejati/BEFUnet.git . 2>&1 || echo 'Repository may already exist or clone completed'

# If you have a specific branch, uncomment and modify:
# !git checkout your-branch-name

# Verify we're in the right place
import os
print(f'Current directory: {os.getcwd()}')
files = [f for f in os.listdir('.') if not f.startswith('.')]
print(f'Files in directory ({len(files)} total): {files[:10]}')

## Install Packages

In [ ]:
!pip install -r requirements.txt

## Download Synapse Dataset

In [ ]:
# Download dataset directly to Drive folder
!gdown 1IGe2kUzBwpMR-HYUotc1pacNjrEo1ftz
!unzip -xq ./data.zip -d . || echo "Dataset already extracted"

## Train Model on Synapse Dataset

**Resume Training Feature:**
- The training automatically saves checkpoints after each epoch
- If training is interrupted, simply run the same command again - it will automatically detect and resume from the latest checkpoint
- Checkpoints are saved in: `./results/BEFUnet/BEFUnet_checkpoint_epoch_X.pth`
- You can also manually specify a checkpoint with `--resume /path/to/checkpoint.pth`

**Note:** Since we're working directly in Drive, all checkpoints and results are automatically saved and persistent!

In [ ]:
# Training will automatically resume from the latest checkpoint if available
# All results are saved directly in Drive - no need to copy anything!
!python train.py \
  --root_path ./data/Synapse/train_npz \
  --test_path ./data/Synapse/test_vol_h5 \
  --batch_size 10 \
  --eval_interval 20 \
  --max_epochs 500 \
  --model_name BEFUnet \
  --num_workers 2 \
  --output_dir ./results

# To manually specify a checkpoint to resume from, add:
# --resume /path/to/checkpoint.pth

## Visualize Results (Optional)

Visualize segmentation results and training curves.

In [ ]:
# Install visualization packages if needed
!pip install nibabel matplotlib pandas

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import os

# Path to example case (adjust as needed)
img_path  = './results/BEFUnet/test/case0001_img.nii.gz'
gt_path   = './results/BEFUnet/test/case0001_gt.nii.gz'
pred_path = './results/BEFUnet/test/case0001_pred.nii.gz'

# Check if files exist
if all(os.path.exists(p) for p in [img_path, gt_path, pred_path]):
    img  = nib.load(img_path).get_fdata()
    gt   = nib.load(gt_path).get_fdata()
    pred = nib.load(pred_path).get_fdata()

    # Select middle slice
    z = img.shape[2] // 2
    img_slice  = img[:,:,z]
    gt_slice   = gt[:,:,z]
    pred_slice = pred[:,:,z]

    # Plot side-by-side
    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    axs[0].imshow(img_slice, cmap='gray')
    axs[0].set_title('Input Image')
    axs[0].axis('off')
    
    axs[1].imshow(gt_slice, cmap='gray')
    axs[1].set_title('Ground Truth')
    axs[1].axis('off')
    
    axs[2].imshow(pred_slice, cmap='gray')
    axs[2].set_title('Prediction')
    axs[2].axis('off')

    plt.tight_layout()
    plt.savefig('example_segmentation.png', dpi=300)
    plt.show()
    
    # Download image
    from google.colab import files
    files.download('example_segmentation.png')
else:
    print("Result files not found. Run training first.")

In [ ]:
# Plot training curves from CSV results
import pandas as pd
import matplotlib.pyplot as plt
import glob

# Find the most recent results CSV file
csv_files = glob.glob('./results/BEFUnet/*results.csv')
if csv_files:
    # Get the most recent file
    latest_csv = max(csv_files, key=os.path.getctime)
    print(f"Loading: {latest_csv}")
    
    df = pd.read_csv(latest_csv, sep='\t')
    print(f"Columns: {df.columns.tolist()}")
    
    # Plot Dice curve
    if 'mean_dice' in df.columns:
        plt.figure(figsize=(10, 5))
        plt.plot(df['mean_dice'], marker='o', linestyle='-', label='Mean Dice')
        if 'mean_hd95' in df.columns:
            plt.plot(df['mean_hd95'], marker='s', linestyle='--', label='Mean HD95')
        plt.xlabel('Epoch')
        plt.ylabel('Score')
        plt.title('Training Metrics')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig('training_curves.png', dpi=300)
        plt.show()
        
        from google.colab import files
        files.download('training_curves.png')
    else:
        print("mean_dice column not found in CSV")
else:
    print("No results CSV file found. Run training first.")